# IPTA DR2 — nonlinear marginalization validation (J1640+2224)

New-API rebuild of the marginalization validation. It shows the two ways to
*analytically marginalize* a timing axis and that they are genuinely distinct,
first-class records — on the real EPTA-DR2 J1640+2224 data:

- **delta-flat** (`marginalize_delta_flat`): an improper flat-in-δ GP;
- **z-prior** (`marginalize_z_prior`): a proper unit-normal GP on the whitened
  coefficient — a *different measure* with a *different fingerprint*.

Requires the EPTA-DR2 J1640 par/tim and the devcontainer stack.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")
from pathlib import Path


import discovery as ds
from discovery import transport as dst
from metapulsar import create_metapulsar
from metapulsar.sandbox_tempo2 import configure_logging
from nltiming import (
    NonLinearTimingModel, TimingInference, WhiteningConfig,
)

ds.config(kernels="metamath")
configure_logging(level="WARNING")

# Real IPTA-DR2 J1640+2224 (EPTA v2.2). Find the repo root from the CWD.
_here = Path.cwd()
_repo = next(p for p in (_here, *_here.parents) if (p / "data" / "ipta-dr2").is_dir())
PAR = _repo / "data/ipta-dr2/EPTA_v2.2/J1640+2224/J1640+2224.par"
TIM = _repo / "data/ipta-dr2/EPTA_v2.2/J1640+2224/J1640+2224_all.tim"
assert PAR.exists(), f"J1640 par not found: {PAR}"

mp = create_metapulsar(
    {"epta": [{"par": str(PAR), "tim": str(TIM), "timing_package": "tempo2"}]},
    use_pulse_numbers="no",
)
nd = {f"{mp.name}_efac": 1.0, f"{mp.name}_log10_t2equad": -8.0}
reference = dst.reference_noise_frozen(
    ds.makenoise_measurement_simple(mp, nd), nd, description="J1640 WN reference")
center = {f"{mp.name}_rednoise_log10_A": -14.0, f"{mp.name}_rednoise_gamma": 3.5}
print(f"{mp.name}: {len(mp.toas)} TOAs, {len(mp.fitpars)} fitpars")
print("fitpars:", list(mp.fitpars))


## 1. Two dispositions for the same subset

Marginalize the dispersion axes two ways. Each fitpar gets exactly one
disposition; `chart_summary` reports it. The plans are **distinct records** — the
likelihood normalization differs (improper flat vs proper unit-normal), so their
fingerprints differ and run products can never be confused.


In [ ]:
subset = [p for p in ("DM", "DM1", "DM2") if p in mp.fitpars]

ctx_df = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(delta_flat=subset), name="timing"
).for_pulsar(mp)
ctx_zp = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(z_prior=subset), name="timing"
).for_pulsar(mp)

print("subset:", subset)
print("delta-flat: marg_delta =", ctx_df.plan.marginalized_delta,
      "| marg_z =", ctx_df.plan.marginalized_z)
print("z-prior   : marg_delta =", ctx_zp.plan.marginalized_delta,
      "| marg_z =", ctx_zp.plan.marginalized_z)
print("\ndistinct plan fingerprints:",
      ctx_df.plan.fingerprint() != ctx_zp.plan.fingerprint())
print("  delta-flat:", ctx_df.plan.fingerprint()[:32])
print("  z-prior   :", ctx_zp.plan.fingerprint()[:32])


## 2. Different measure, different likelihood

Internally the delta-flat block is an **improper** GP (`makegp_improper`,
constant 1e40) and the z-prior block is a **proper** unit-normal GP
(`makegp_standard_normal`) — a different measure, so the analytically
marginalized log-likelihoods differ even at the same timing point (delta keys at
the par-file reference, red noise integrated at a fixed hyper).


In [ ]:
nd_hyper = {**nd, f"{mp.name}_rednoise_log10_A": -14.0, f"{mp.name}_rednoise_gamma": 3.5}
for label, ctx_m in (("delta-flat", ctx_df), ("z-prior", ctx_zp)):
    psl = ds.PulsarLikelihood([
        mp.residuals, ds.makenoise_measurement_simple(mp, nd),
        ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
        *ctx_m.discovery_signals(joint=False)])
    # timing delay keys at the reference (delta=0); marginalized blocks integrated.
    params = dict(nd_hyper)
    for k in ctx_m.delay_keys:
        params[k] = 0.0
    print(f"{label:10s} logL(reference) = {float(psl.logL(params)):.6g}")
print("\n-> the two marginalizations are distinct models: the improper flat-in-δ")
print("   measure and the proper unit-normal measure give different evidence.")


## 3. z-prior coefficients are sampleable (Enterprise)

A `marginalize_z_prior` block can be *integrated* (default) or *sampled* — pass
`sample_z_coefficients=True` to promote it to a `GPCoefficients` parameter with a
unit-normal prior, for a jointly-sampled / decentered workflow. Here the three
DM-family z-prior coefficients appear as sampled Enterprise parameters.


In [ ]:
from enterprise.signals import parameter, signal_base, white_signals

ntm_zp = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(z_prior=subset),
    whitening=WhiteningConfig(), name="timing")
white = white_signals.MeasurementNoise(efac=parameter.Constant(1.0))
pta = signal_base.PTA([(white + ntm_zp.enterprise_signal(sample_z_coefficients=True))(mp)])
coeff = [p for p in pta.param_names if "zprior_coefficients" in p]
print("z-prior coefficient parameters:", coeff)
print("all timing parameters:", [p for p in pta.param_names if "timing" in p])


## Summary

- `marginalize_delta_flat` and `marginalize_z_prior` are **first-class, distinct**
  per-axis choices: different GP (improper vs proper unit-normal), different
  likelihood normalization, different plan fingerprint — never silently
  interchangeable.
- A z-prior block is integrated by default but can be **sampled**
  (`sample_z_coefficients=True`) when the workflow wants those coefficients in the
  joint / decentered vector.
- Disposition (sample / delta-flat / z-prior) is **orthogonal** to identical
  linearity — the geometry properties of the sampled axes are validated
  separately in `03_j1640_decentering_validation.ipynb`.
